<a href="https://colab.research.google.com/github/tiryakiselvi/my-first-project/blob/main/01_BPE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vertiefung der Grundlagen der CL WS26/27
    

## Exercise 01 - BPE Tokenization  

**Submission deadline**: 27.10.2026, 11:59 a.m.

**You will only receive feedback if you submit the notebook before the deadline.**

**Instruction for submission**: You should complete the parts of the exercises that are marked as **TODO** (⚠️ but don't remove the `TODO` comments! ⚠️). After completing the exercise, save a copy of the notebook as `ex01_BPE_[MATRIKELNUMMER].ipynb`, where `[MATRIKELNUMMER]` is your matriculation number. Then submit the notebook on Moodle.

------

After completing this assignment, you will:

- understand how the Byte-Pair Encoding (BPE) algorithm works
- be able to implement BPE for tokenization
- use the `tokenizers` library to train a BPE tokenizer

## Byte-Pair Encoding for Tokenization
**Byte-Pair Encoding (BPE)** ([Sennrich et al., 2016](https://aclanthology.org/P16-1162.pdf)) is a *subword-based* tokenization algorithm.
Instead of defining tokens as words, or as characters, we can use our data to automatically tell us what the tokens should be.
Modern tokenizers often automatically induce sets of tokens that include tokens smaller than words, called **subwords**, in order to deal with unknown word problem.

Most tokenization schemes have two parts: a **token learner**, and a **token segmenter**. BPE algorithm is one of the most simple and commonly used subword-based tokenization algorithms. The main idea of BPE is as follows:
1. BPE **token learner** starts with a vocabulary that is just the set of all individual characters
2. Examines the training corpus and chooses the two symbols that are most frequently adjacent (say `'A'`, `'B'`)
3. Adds a new merged symbol `'AB'` to the vocabulary
4. Replaces every adjacent `'A'`, `'B'` in the corpus with `'AB'`.
5. Repeat 2-4, until the vocabulary size reaches `k`, which is a parameter of the algorithm.

Once we've learned our vocabulary, the **token segmenter** is used to tokenize a sentence. The token segmenter just runs the merges we have learned from the training corpus greedily in the order we learned them.

**Install required libraries**  

In [ ]:
%pip install datasets
%pip install tokenizers

## Task 1: Implement the BPE algorithm with a toy example
Next, let's use a toy example to see how BPE algorithm is implemented in Python step by step.

#### Prepare the toy training corpus

In [ ]:
from collections import defaultdict

# end-of-word token
EOW = '#'

# desired vocabulary size
VOCAB_SIZE = 15  # vocabulary size is the only hyperparameter of the BPE algorithm.

# toy training corpus
corpus = 'hug ' * 10 + 'pug ' * 5 + 'pun ' * 12 + 'bun ' * 4 + 'hugs ' * 5

corpus

'hug hug hug hug hug hug hug hug hug hug pug pug pug pug pug pun pun pun pun pun pun pun pun pun pun pun pun bun bun bun bun hugs hugs hugs hugs hugs '

#### Step 1 - Initialization

* **Convert the corpus to word counts.**

In [ ]:
def init_words(corpus):
    """
    Count the frequency of each whitespace-separated word in the corpus.

    Output:
    {'hug': 10, 'pug': 5, 'pun': 12, 'bun': 4, 'hugs': 5}
    """
    words = corpus.split()
    return {
        word: words.count(word)
        for word in set(words)
    }

words = init_words(corpus)

* **Initialize the vocabulary and the merge rules.**

In [ ]:
# Initialize the vocabulary

def init_vocab(words):
    """
    Input:
    words = {'hug': 10, 'pug': 5, 'pun': 12, 'bun': 4, 'hugs': 5}

    Output:
    {'#', 'p', 'n', 'b', 'g', 'h', 'u', 's'}
    """

    vocab = set()

    # TODO 1: initialize the vocabulary as a set with all characters in the corpus
    # and the end-of-word (EOW) character

    for word in words:
        for char in word:
            vocab.add(char)
    vocab.add(EOW)

    # End of TODO 1

    return vocab

vocab = init_vocab(words)
print(vocab)

# Initialize the merge rules.

merges = []  # Keep track of the merge operations in the order they were applied

{'g', 'n', '#', 'p', 's', 'h', 'b', 'u'}


* **Convert the corpus into the tokenized form**  
Before starting to train, we need to first split each word into individual characters.
We use a `dict` that maps each word to a list containing its tokenized form and its frequency, serving as the representation of the entire corpus.

In [ ]:
def init_tokens(words):
    """
    Example:

    Input:
    words = {'hug': 10, 'pug': 5, 'pun': 12, 'bun': 4, 'hugs': 5}

    Output:
    {'hug': [['h', 'u', 'g', '#'], 10],
     'pug': [['p', 'u', 'g', '#'], 5],
     'pun': [['p', 'u', 'n', '#'], 12],
     'bun': [['b', 'u', 'n', '#'], 4],
     'hugs': [['h', 'u', 'g', 's', '#'], 5]}
    """

    tokens = {}

    # TODO 2: initialize the tokenized corpus

    for word, freq in words.items():
        tokens[word] = [list(word) + [EOW], freq]

    # End of TODO 2

    return tokens

tokenized_words = init_tokens(words)

tokenized_words

{'hug': [['h', 'u', 'g', '#'], 10],
 'pun': [['p', 'u', 'n', '#'], 12],
 'hugs': [['h', 'u', 'g', 's', '#'], 5],
 'bun': [['b', 'u', 'n', '#'], 4],
 'pug': [['p', 'u', 'g', '#'], 5]}

#### Step 2 - Compute the frequency for all character pairs


In [ ]:
# count cooccurences of all characters
def get_coocs(tokenized_words):
    """
    Example:

    Input:
    {'hug': [['h', 'u', 'g', '#'], 10],
    'pug': [['p', 'u', 'g', '#'], 5],
    'pun': [['p', 'u', 'n', '#'], 12],
    'bun': [['b', 'u', 'n', '#'], 4],
    'hugs': [['h', 'u', 'g', 's', '#'], 5]}

    Output:
    defaultdict(int,
                {('h', 'u'): 15,
                ('u', 'g'): 20,
                ('g', '#'): 15,
                ('p', 'u'): 17,
                ('u', 'n'): 16,
                ('n', '#'): 16,
                ('b', 'u'): 4,
                ('g', 's'): 5,
                ('s', '#'): 5})

    """
    coocs = defaultdict(int)

    # TODO 3: Compute the frequency for all character pairs

    for word in tokenized_words:
        tokens, freq = tokenized_words[word]
        for i in range(len(tokens) - 1):
            coocs[(tokens[i], tokens[i + 1])] += freq

    # End of TODO 3

    return coocs

coocs = get_coocs(tokenized_words)
coocs

defaultdict(int,
            {('h', 'u'): 15,
             ('u', 'g'): 20,
             ('g', '#'): 15,
             ('p', 'u'): 17,
             ('u', 'n'): 16,
             ('n', '#'): 16,
             ('g', 's'): 5,
             ('s', '#'): 5,
             ('b', 'u'): 4})

#### Step 3 - Update merge rules and the vocabulary.

Find the most frequent character pair:

In [ ]:
# TODO 4: Replace ... with your own code.

# most frequent pair (on ties take the alphabetically larger one, so the result is reproducible)
best_pair = max(coocs, key=lambda pair: (coocs[pair], pair))

# End of TODO 4

best_pair


('u', 'g')

Update the merge rules:

In [ ]:
merges.append(best_pair)
merges

[('u', 'g')]

Update the vocabulary

In [ ]:
new_token = ''.join(best_pair)
vocab.add(new_token)
vocab

{'#', 'b', 'g', 'h', 'n', 'p', 's', 'u', 'ug'}

#### Step 4 - Update the split corpus

In [ ]:
# Apply a merge to the split corpus (replace all pair occurrences with the merged token)
def update_corpus(tokenized_words, pair):
    """
    Example:

    Input:
    tokenized_words = {'hug': [['h', 'u', 'g', '#'], 10],
                       'pug': [['p', 'u', 'g', '#'], 5],
                       'pun': [['p', 'u', 'n', '#'], 12],
                       'bun': [['b', 'u', 'n', '#'], 4],
                       'hugs': [['h', 'u', 'g', 's', '#'], 5]}
    pair = ('u', 'g')

    Output:
    {'hug': [['h', 'ug', '#'], 10],
     'pug': [['p', 'ug', '#'], 5],
     'pun': [['p', 'u', 'n', '#'], 12],
     'bun': [['b', 'u', 'n', '#'], 4],
     'hugs': [['h', 'ug', 's', '#'], 5]}
    """

    # TODO 5: update the corpus

    left, right = pair
    merged = left + right

    for word, (tokens, freq) in tokenized_words.items():
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == left and tokens[i + 1] == right:
                new_tokens.append(merged)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokenized_words[word] = [new_tokens, freq]

    # End of TODO 5

    return tokenized_words

tokenized_words = update_corpus(tokenized_words, best_pair)
tokenized_words

{'hug': [['h', 'ug', '#'], 10],
 'pun': [['p', 'u', 'n', '#'], 12],
 'hugs': [['h', 'ug', 's', '#'], 5],
 'bun': [['b', 'u', 'n', '#'], 4],
 'pug': [['p', 'ug', '#'], 5]}

#### Step 5 - Repeat 2-4 until vocabulary size is achieved

Now, let's put everything together!

In [ ]:
def train_tokenizer(corpus, vocab_size):
    # Initialization:
    words = init_words(corpus)
    vocab = init_vocab(words)
    tokenized_words = init_tokens(words)
    merges = []

    while len(vocab) < vocab_size:

        # TODO 6: Use the functions and snippets to repeat steps 2-4
        # until the vocabulary reaches the desired size.

        coocs = get_coocs(tokenized_words)

        # no pairs left -> nothing more to merge
        if not coocs:
            break

        best_pair = max(coocs, key=lambda pair: (coocs[pair], pair))
        merges.append(best_pair)
        vocab.add(''.join(best_pair))
        tokenized_words = update_corpus(tokenized_words, best_pair)

        # End of TODO 6

    return vocab, merges

vocab_size = 15
vocab, merges = train_tokenizer(corpus, vocab_size)
print(vocab)
merges

{'g', 'hug#', 'n', '#', 'pun#', 'un#', 'ugs', 'p', 's', 'ug#', 'ug', 'un', 'h', 'b', 'u'}


[('u', 'g'),
 ('u', 'n'),
 ('un', '#'),
 ('ug', '#'),
 ('p', 'un#'),
 ('h', 'ug#'),
 ('ug', 's')]

## Task 2: Implement BPE tokenization on a large corpus

In this task, we will employ the BPE algorithm implementation of the previous task for a large corpus.
Then, we apply all the merge rules learned to tokenize some sentences.

#### Step 1 - Prepare the corpus
For this task, we extract a multilingual corpus from the [XNLI](https://huggingface.co/datasets/facebook/xnli) dataset provided by the Huggingface.
The XNLI dataset can be loaded using the `datasets` library.  
We use a subset of the XNLI training set to form the corpus for the training of a BPE tokenizer.

In [ ]:
from datasets import load_dataset

# load the XNLI dataset
langs = ['en', 'bg', 'zh'] # English, Bulgarian, Chinese
corpora = []

for lang in langs:
    xnli = load_dataset('facebook/xnli', lang)

    # build a corpus from the subset of the XNLI training set
    corpus = " ".join(text for text in xnli['train']['hypothesis'][:500])
    corpora.append(corpus)
full_corpus = " ".join(corpora)

# Represent the corpus using the word frequency
words = init_words(full_corpus)
len(words)

README.md:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

en/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 50.2MB            

en/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

en/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  308kB            

en/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

zh/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/392702 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5010 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2490 [00:00<?, ? examples/s]

5868

#### Step 2 - Run the BPE algorithm on the XNLI dataset
Here we use your code from Task 1 to implement the BPE algorithm on the XNLI dataset.

In [ ]:
vocab_size = 6000

vocab, merges = train_tokenizer(full_corpus, vocab_size)

len(vocab), len(merges)

(6000, 4600)

#### Step 3 - Apply the learned merge rules to tokenization


In [ ]:
# Use merge rules to tokenize a given text
def tokenize(text, merges):
    """
    Example:
        Input:
            "This is an easy example."
        return
            ['This#', 'is#', 'an#', 'easy#', 'ex', 'am', 'ple', '.#']
    """

    # We first pre-tokenize the text at whitespaces
    words = text.split()

    # Initialize tokenization with character-level tokens and the end-of-word token
    tokenized_words = []
    for word in words:
        tokenized_words.append(list(word)+[EOW])

    for pair in merges:

        # TODO: Apply the merge rules to the tokenized words in the same order

        left, right = pair
        merged = left + right

        for idx, tokens in enumerate(tokenized_words):
            new_tokens = []
            i = 0
            while i < len(tokens):
                if i < len(tokens) - 1 and tokens[i] == left and tokens[i + 1] == right:
                    new_tokens.append(merged)
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1
            tokenized_words[idx] = new_tokens

        # End of TODO

    # Combine the tokenized words into a single list of tokens
    tokenized_text = sum(tokenized_words, [])
    return tokenized_text

text0 = "This is an easy example."
text1 = "продукт и география са това , което прави крем краде работа."
text2 = "Use BPE algorithm to tokenize a sample sentence."
text3 = "今天 工作 有 一点 累"
print(tokenize(text0, merges))
print(tokenize(text1, merges))
print(tokenize(text2, merges))
print(tokenize(text3, merges))

['This#', 'is#', 'an#', 'easy#', 'ex', 'a', 'mp', 'le', '.#']
['продук', 'т#', 'и#', 'г', 'е', 'огра', 'фи', 'я#', 'са#', 'това#', ',#', 'което#', 'прави#', 'к', 'ре', 'м#', 'кра', 'де#', 'работ', 'а', '.#']
['U', 'se#', 'B', 'P', 'E', '#', 'al', 'g', 'or', 'i', 'th', 'm#', 'to#', 'to', 'k', 'en', 'iz', 'e#', 'a#', 'sa', 'mp', 'le#', 'sen', 't', 'enc', 'e', '.#']
['今', '天#', '工作#', '有#', '一', '点#', '累', '#']


## Task 3: Use `tokenizers` to train a BPE tokenizer
In this task, we will use the `tokenizers` library from Hugging Face, which is commonly used for large language models.

#### Step 1 - Instantiate the tokenizer and trainer

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer

VOCAB_SIZE = 6000

# TODO 1: Instantiate the tokenizer with whitespace pre-tokenization
# and a trainer to define the vocabulary size.
# Documentation: https://huggingface.co/docs/tokenizers/v0.23.3/en/index)

bpe_tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
bpe_tokenizer.pre_tokenizer = Whitespace()
bpe_trainer = BpeTrainer(vocab_size=VOCAB_SIZE, special_tokens=["[UNK]"])

# End of TODO 1

#### Step 2 - Train the BPE models

In [ ]:
full_corpus.split()
bpe_tokenizer.train_from_iterator([full_corpus], trainer=bpe_trainer)
bpe_tokenizer.get_vocab_size()

6000

#### Step 3 - Use the trained model for tokenization

In [ ]:
"""
text0 = "This is an easy example."
text1 = "продукт и география са това , което прави крем краде работа."
text2 = "Use BPE algorithm to tokenize a sample sentence."
text3 = "今天 工作 有 一点 累"
"""

print(bpe_tokenizer.encode(text0).tokens)
print(bpe_tokenizer.encode(text1).tokens)
print(bpe_tokenizer.encode(text2).tokens)
print(bpe_tokenizer.encode(text3).tokens)

['This', 'is', 'an', 'easy', 'ex', 'am', 'ple', '.']
['проду', 'кт', 'и', 'ге', 'огра', 'фи', 'я', 'са', 'това', ',', 'което', 'прави', 'кре', 'м', 'кра', 'де', 'работа', '.']
['U', 'se', 'B', 'P', 'E', 'al', 'gor', 'ith', 'm', 'to', 'to', 'k', 'en', 'iz', 'e', 'a', 's', 'am', 'ple', 's', 'ent', 'ence', '.']
['今天', '工作', '有', '一点', '[UNK]']


## Conceptual questions

1. Compare the tokenization results of Task 2 and 3 and analyse their differences. What could be possible reasons that cause the differences?

> The biggest difference I can see in the outputs is the #. In my tokenizer every word ends with #, so I get tokens like is#, while the tokenizers library does not mark the end of a word at all. So in mine the same string can be two different tokens (at the end of a word and inside a word), in the library it is only one. Another reason is the pre-tokenization. I only split at spaces, so a full stop stays attached to the word before it ("example.") and can be merged with it. The Whitespace pre-tokenizer of the library cuts off punctuation first, so punctuation is always its own token. That changes the word counts and so also which pairs get merged first. Unknown characters are also handled differently: in my version they just stay single characters, the library turns them into [UNK]. On top of that, the vocabulary size is counted a bit differently, and with only about 1500 sentences and a vocabulary of 6000 many merges have a frequency of just 1 or 2, so which one comes first depends on how each implementation breaks ties. I would expect frequent words to be tokenized similarly in both and rare words to be split differently.

2. What would happen if one language was underrepresented in the corpus that used to train a multilingual tokenizer? How would this affect the tokenization for this language?

> The tokenizer only learns merges that are frequent in the training data. If a language hardly appears in the corpus, its character combinations are rare and almost never become the most frequent pair, so most of the merges go to the big language(s). For the small language only a few merges are learned, and its words end up split into very short pieces, often single characters or even bytes. Characters that were never seen in training can become unknown tokens. This means the same sentence needs many more tokens than in English, which makes it more expensive and slower and leaves less room in the context window. The tokens also carry hardly any meaning and are rare in the training data, so the model learns worse representations for this language and will probably perform worse on it. Training the tokenizer on more balanced data or using a larger vocabulary would help.

3. What are the advantages and disadvantages of using a smaller vocabulary size? Under which circumstances might downstream performance benefit from this?

> A smaller vocabulary means a smaller embedding matrix and output layer, so fewer parameters and less memory. Each token is also seen more often in training, so its embedding is trained better, and rare or new words can still be built from known subwords. The downside is that texts become longer in tokens, because words are split into more pieces. This costs more computation (attention grows quadratically with the sequence length), less text fits into the context window, and a single token has less meaning, so the model has to put words together from pieces. I think a smaller vocabulary can help when there is little training data, for languages with a lot of morphology, for noisy text with typos and rare words, or when the model is small and memory is limited.